# Samplers and schedules

[Notebook 02](02-train-a-diffusion-model.ipynb) trained a diffusion model and sampled it with one solver at one step count. This notebook restores that checkpoint and compares every solver Dew ships: DDPM, DDIM, Euler, Euler ancestral, Heun, RK4 and MultiStepDPM, each at several step counts, from the same starting noise, with the wall time each takes. It also shows what a preset pairs up, because a solver only walks the trajectory the training process defined.

Run notebook 02 first: this one reads `runs/02-diffusion` from the same working directory, on whichever route notebook 02 used. The offline route takes a couple of minutes on a CPU.

In [ ]:
# On Colab: install dew and the JAX build for the runtime. Locally this cell is a no-op.
try:
    import google.colab  # noqa: F401
    import subprocess, sys
    try:
        import jax
        tpu = any("tpu" in str(d).lower() for d in jax.devices())
    except Exception:
        tpu = False
    extra = "jax[tpu]" if tpu else "jax[cuda12]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "dew-ml[tfds] @ git+https://github.com/AshishKumar4/dew", extra])
except ImportError:
    pass

In [ ]:
ROUTE = "offline"          # must match notebook 02
CHECKPOINT = "runs/02-diffusion"

if ROUTE == "offline":
    IMAGE_SIZE = 16
    MODEL = dict(patch_size=4, emb_features=32, num_layers=2, num_heads=2, mlp_ratio=2)
    DTYPE, ATTENTION = "float32", "xla"
    STEP_COUNTS = (4, 8, 16)
else:
    IMAGE_SIZE = 64
    MODEL = dict(patch_size=4, emb_features=384, num_layers=8, num_heads=6)
    DTYPE, ATTENTION = "bfloat16", "auto"
    STEP_COUNTS = (10, 20, 50)
SAMPLES = 8
SEED = 0

In [ ]:
import jax

print(jax.devices())
print(jax.default_backend())

## Rebuild the model and restore the checkpoint

A checkpoint holds arrays, not code. The model is rebuilt from the same fields notebook 02 used and the process from the same preset, then `Checkpoints.restore` reads the latest step. The template names which leaves to read and where to put them: the abstract shapes from `objective.init`, placed on one device since sampling runs on one process, and the EMA subtree the objective selects. `merge` puts the EMA leaves in place of the live ones, which is what `state.averaged` did in notebook 02.

In [ ]:
import jax.numpy as jnp
from dew import Checkpoints, Field, InputSpec, models, presets
from dew.objectives.base import merge, select
from dew.objectives.diffusion import DiffusionObjective

process = presets.EDM()()
model = models.build("simple_dit", **MODEL, output_channels=3, dtype=DTYPE, attention_impl=ATTENTION)
inputs = InputSpec(Field("image", (IMAGE_SIZE, IMAGE_SIZE, 3)))
objective = DiffusionObjective(model, process, inputs, ema_decay=0.99)

device = jax.sharding.SingleDeviceSharding(jax.devices()[0])
abstract = jax.tree.map(
    lambda leaf: jax.ShapeDtypeStruct(leaf.shape, leaf.dtype, sharding=device),
    jax.eval_shape(objective.init, jax.random.key(0)))
template = {"params": abstract, "ema": select(abstract, objective.ema.select)}
values, _ = Checkpoints(CHECKPOINT).restore(template)
params = merge(values["params"], values["ema"])
step = int(Checkpoints(CHECKPOINT).latest)
print(f"restored the EMA weights ({len(jax.tree_util.tree_leaves(params))} arrays) from step {step}")

## The same starting noise for everyone

Every solver integrates the same trajectory, so the only fair comparison fixes everything but the solver. `process.noise` draws x_T with the variance the schedule's top noise level implies; drawing it once from one key and handing it to every run keeps the comparison honest. Ancestral samplers still inject their own noise along the way, which is part of what is being compared, and `sample` folds the run's key into each step for that.

In [ ]:
x_T = process.noise(jax.random.key(SEED), (SAMPLES, IMAGE_SIZE, IMAGE_SIZE, 3))
print("x_T std:", float(jnp.std(x_T)))

## The solvers

- **DDPM** samples the reverse Markov chain the schedule defines. It needs small steps: at few steps it has not converged and shows unremoved noise.
- **DDIM** is DDPM's deterministic shortcut: jump to any next noise level, keep the direction the model implies.
- **Euler** integrates the probability-flow ODE with one evaluation per step. On the Karras schedule it lands very close to DDIM.
- **Euler ancestral** solves the reverse SDE: after every deterministic step it injects a calibrated fraction of fresh noise, so each run differs.
- **Heun** is Euler with a correction: evaluate at the end of the step, average the two slopes. Two evaluations per step buy roughly second-order accuracy.
- **RK4** is classical fourth-order Runge-Kutta: four evaluations per step.
- **MultiStepDPM** reuses predictions from earlier steps for higher-order corrections without extra evaluations.

Every solver is a value from `dew.sampling`, and `sample(denoise, x_T, steps, solver=..., key=...)` runs it. `denoise` comes from `process.denoiser`, which carries the process the solver reads.

In [ ]:
import time

import numpy as np
from PIL import Image

try:
    from IPython.display import display
except ModuleNotFoundError:  # running the cells as a plain script
    def display(image):
        print(f"image {image.width}x{image.height}")

from dew.sampling import (DDIM, DDPM, Euler, EulerAncestral, Heun, MultiStepDPM, RK4,
                          sample)

SOLVERS = {
    "DDPM": DDPM(), "DDIM": DDIM(), "Euler": Euler(), "Euler-A": EulerAncestral(),
    "Heun": Heun(), "RK4": RK4(), "MultiStepDPM": MultiStepDPM(),
}
denoise = process.denoiser(model, objective.trainable(params), {})


def show_grid(frames, cols, scale=4):
    frames = np.asarray(frames)
    rows = (len(frames) + cols - 1) // cols
    h, w, c = frames.shape[1:]
    grid = frames.reshape(rows, cols, h, w, c).transpose(0, 2, 1, 3, 4).reshape(rows * h, cols * w, c)
    image = Image.fromarray(grid)
    display(image.resize((image.width * scale, image.height * scale), Image.NEAREST))
    return image


timings, images = {}, {}
for n_steps in STEP_COUNTS:
    for name, solver in SOLVERS.items():
        start = time.perf_counter()
        out = sample(denoise, x_T, n_steps, solver=solver, key=jax.random.key(SEED))
        out.block_until_ready()
        timings[(name, n_steps)] = time.perf_counter() - start
        images[(name, n_steps)] = np.asarray(out)
    print(f"{n_steps} steps: rows are {', '.join(SOLVERS)}")
    frames = np.clip((np.concatenate([images[(name, n_steps)] for name in SOLVERS]) + 1) * 127.5,
                     0, 255).astype(np.uint8)
    show_grid(frames, cols=SAMPLES)

In [ ]:
print(f"{'solver':<13}" + "".join(f"{n:>9}" for n in STEP_COUNTS))
for name in SOLVERS:
    print(f"{name:<13}" + "".join(f"{timings[(name, n)]:>8.2f}s" for n in STEP_COUNTS))
evals = {"DDPM": 1, "DDIM": 1, "Euler": 1, "Euler-A": 1, "Heun": 2, "RK4": 4, "MultiStepDPM": 1}
print()
print(f"{'evals/step':<13}" + "".join(f"{evals[name]:>9}" for name in SOLVERS))

## What the grids and the numbers say

Each first call includes XLA compilation, so the timings are compile plus sampling; the second row of a repeated cell shows the sampling cost alone. The wall time otherwise tracks evaluations per step: RK4 costs about four Euler runs, Heun two, MultiStepDPM about one.

Euler and DDIM should land within noise of each other at every count, the algebraic equivalence showing through. The cell below measures the pairwise distance between each solver's samples and Euler's at the largest step count; the deterministic solvers converge toward one another as steps grow, while the ancestral ones keep their own noise.

In [ ]:
reference = images[("Euler", STEP_COUNTS[-1])]
for name in SOLVERS:
    distance = float(np.mean(np.abs(images[(name, STEP_COUNTS[-1])] - reference)))
    print(f"{name:<13} mean |x - Euler| at {STEP_COUNTS[-1]} steps: {distance:.4f}")

## The presets behind the pairings

A training schedule and a sampling schedule are not independent choices. A preset builds one `Process` holding the training schedule, the prediction transform, the loss weighting and, where it differs, the sampling schedule: EDM trains on a log-normal sigma distribution and samples on the Karras spacing; the cosine preset trains and samples a discrete variance-preserving schedule with v-prediction; the flow preset is rectified flow on both sides. Sampling a model with a different process than it trained under produces the wrong trajectory, which is why one object carries both halves.

In [ ]:
for name in ("edm", "karras", "cosine", "flow"):
    built = presets.build(name)()
    print(f"{name:<8} train={type(built.schedule).__name__:<28} "
          f"sample={type(built.sampler_schedule).__name__:<28} "
          f"prediction={type(built.prediction).__name__}")

## Where to go next

With a trained model, a checkpoint and a sense of which solver fits your budget, [notebook 03](03-text-to-image-with-guidance.ipynb) adds text conditioning and classifier-free guidance on top of the same `sample` call.